# 04 · Experimentos del recomendador por canción

**Proyecto:** Spotify Music Intelligence
**Módulo 4:** Recomendador por canción
**Objetivo:** comparar las cuatro configuraciones obligatorias (R1–R4,
AGENTS.md §14.4) sobre semillas manuales sin usar test y sin modificar datos.

| ID | Escalador | Distancia |
|---|---|---|
| R1 | StandardScaler | coseno |
| R2 | RobustScaler | coseno |
| R3 | StandardScaler | euclídea |
| R4 | RobustScaler | euclídea |

Baseline inicial: `StandardScaler + NearestNeighbors(cosine, brute)`.

Unidad de modelado: `recording_group_id`. Se excluyen las grabaciones con
análisis acústico incompleto y se excluye siempre la propia grabación.

## Configuración y carga

Se cargan los datos procesados y el código reutilizable del paquete
`spotify_intelligence.recommenders`. Ningún cálculo pesado vive en este
notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd

from spotify_intelligence.recommenders import experiments as exp

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

recordings = pd.read_parquet("data/processed/recordings.parquet")
eligible = recordings[~recordings["audio_analysis_incomplete"]].reset_index(drop=True)

print("recordings:", recordings.shape)
print("elegibles:", eligible.shape)

recordings: (83881, 24)
elegibles: (83736, 24)


## 1. Semillas manuales

Se resuelven canciones reconocibles por `track_name` + `artists` a su
`recording_group_id` dentro del catálogo elegible. La resolución exacta se
simula con búsqueda por subcadena; en la aplicación se usará un buscador
desambiguado.

In [2]:
seed_queries = [
    ("Blinding Lights", "The Weeknd"),
    ("Bohemian Rhapsody - Remastered 2011", "Queen"),
    ("Smells Like Teen Spirit", "Nirvana"),
    ("Easy On Me", "Adele"),
    ("Me Porto Bonito", "Bad Bunny;Chencho Corleone"),
]


def resolve_seed(tracks, query_name, query_artist):
    match = tracks[
        tracks["track_name"].str.contains(query_name, case=False, na=False)
        & tracks["artists"].str.contains(query_artist, case=False, na=False)
    ]
    if match.empty:
        return None
    best = match.sort_values("popularity_median", ascending=False).iloc[0]
    return int(best.name)


seed_rows = []
for name, artist in seed_queries:
    row = resolve_seed(eligible, name, artist)
    if row is None:
        print(f"{name} | {artist} -> NO resuelto")
        continue
    seed_rows.append(row)
    rec = eligible.iloc[row]
    print(f"{name} | {artist} -> {rec['recording_group_id']}")

print("\nSeed rows:", seed_rows)

Blinding Lights | The Weeknd -> a2f99e089dd43c6af5c96cf5ff96c035496f34d9cd2653f815140bcb46ca6a84
Bohemian Rhapsody - Remastered 2011 | Queen -> f8ee6e123e9638931ed45eea2d635e3d5168c613bee551e3b2e71ec301cd18aa


Smells Like Teen Spirit | Nirvana -> 1613efba57c852b695957e4ded8ea9ae9b703b2c5063daaa2566e8aa4318bc34


Easy On Me | Adele -> 2fb7b45f04b9ada55e5ae5224fea9200d4158a9d492710f5bc82618b85acca63
Me Porto Bonito | Bad Bunny;Chencho Corleone -> 5592d47b346d544ecf65c9867186bd57de27e3e1bc69bb482973319ec0a7b8f2

Seed rows: [52999, 81411, 7112, 15395, 27674]


**Resultado:** las cinco semillas se resolvieron a `recording_group_id`.
Estas semillas se usan únicamente para inspección manual y comparación de
configuraciones; no son un ground truth de preferencias.

## 2. Baseline R1

**Método:** `StandardScaler + NearestNeighbors(metric="cosine", algorithm="brute")`
con `Top-N = 10` y primer tramo de recuperación de 100 candidatos.

In [3]:
r1 = exp.run_experiment(
    eligible,
    seed_rows,
    scaler_name="standard",
    metric="cosine",
    top_n=10,
    candidate_floor=100,
)
print(pd.Series(r1).to_string())

id                             R1
scaler                   standard
metric                     cosine
comparable_similarity        True
queries                         5
total_results                  50
self_recommendations            0
duplicate_groups                0
mean_similarity          0.979433
catalog_coverage         0.000597
latency_total_s             0.073
latency_mean_ms            14.597


In [4]:
r1_scaled, r1_nn = exp.build_experiment_index(eligible, scaler_name="standard", metric="cosine")
for seed_row in seed_rows[:2]:
    rows, dists = exp.recommend_with_index(r1_scaled, r1_nn, seed_row, top_n=5)
    names = eligible.iloc[rows][["track_name", "artists"]].values.tolist()
    print(f"\nSeed: {eligible.iloc[seed_row]['track_name']} | {eligible.iloc[seed_row]['artists']}")
    for (name, artist), dist in zip(names, dists, strict=False):
        print(f"  sim={1 - dist:.4f}  {name} | {artist}")


Seed: Blinding Lights | The Weeknd
  sim=0.9923  平凡人的自傳 - Rap Version | ONE PROMISE
  sim=0.9923  Viah | Jass Manak
  sim=0.9890  Thinkin About | ShockOne;Lee Mvtthews
  sim=0.9875  Fool Yourself | Chase & Status;Plan B;Rage
  sim=0.9874  BODY | LICK;LUNA AURA

Seed: Bohemian Rhapsody - Remastered 2011 | Queen
  sim=0.9875  The Passion (Live Acoustic) - Bonus | Hillsong Worship;Brooke Ligertwood
  sim=0.9720  Ana's Song (Open Fire) - Acoustic Re-Mix | Silverchair
  sim=0.9706  Ele Vive - Ao Vivo | Leonardo Gonçalves
  sim=0.9668  Tourner dans le vide - Version Orchestrale | Indila
  sim=0.9629  あなたのすべてになりたい | Seiko Matsuda


**Resultado:** las listas del baseline R1 son coherentes: artistas y
géneros afines a cada semilla, sin autorrecomendación y sin grupos repetidos en
las inspecciones. `similarity = 1 - cosine_distance` y **no es una
probabilidad**.

## 3. Comparación R1–R4

**Método:** se ejecutan las cuatro configuraciones sobre las mismas semillas y
se miden autorrecomendaciones, duplicados, similitud media, cobertura del
catálogo y latencia. El reporte se guarda en
`reports/experiments/track_recommender_r1_r4_comparison.json`.

In [5]:
summary = exp.run_all_experiments(eligible, seed_rows, top_n=10, candidate_floor=100)
summary

,id,scaler,metric,comparable_similarity,queries,total_results,self_recommendations,duplicate_groups,mean_similarity,catalog_coverage,latency_total_s,latency_mean_ms
0,R1,standard,cosine,True,5,50,0,0,0.979433,0.000597,0.0729,14.586
1,R2,robust,cosine,True,5,50,0,0,0.967960,0.000597,0.0770,15.407
2,R3,standard,euclidean,False,5,50,0,0,0.496930,0.000597,1.6934,338.673
3,R4,robust,euclidean,False,5,50,0,0,0.569633,0.000597,0.0346,6.913


In [6]:
path = exp.save_experiment_report(summary, output_dir="reports/experiments")
print("Reporte guardado en", path)

Reporte guardado en reports\experiments\track_recommender_r1_r4_comparison.json


**Resultado:** las cuatro configuraciones cumplen `autorrecomendación = 0`
y `duplicados de grupos = 0`. La similitud media de R1 (coseno + StandardScaler)
es la más alta de las configuraciones coseno (0,979 vs 0,968 de R2). Para las
configuraciones euclídeas (R3, R4) la columna `mean_similarity` usa `1 - distance`
pero la distancia euclídea no está acotada en [0, 1], por lo que ese valor **no
es comparable** con el de las configuraciones coseno; lo que sí es comparable
es la latencia y el cumplimiento de autorrecomendación/duplicados. La latencia
media por consulta con el catálogo completo es de ~15 ms en R1/R2; R4 (euclídea
+ RobustScaler) también es rápida (decenas de ms) mientras que R3 (euclídea +
StandardScaler) suele ser la más lenta (varios cientos de ms). Los valores
exactos se guardan en
`reports/experiments/track_recommender_r1_r4_comparison.json` y varían según el
equipo y la carga.

**Limitación:** las métricas de cobertura sobre 5 semillas no representan la
cobertura real del catálogo; la evaluación completa (módulo 4, evaluación
automática) usa una muestra más amplia. `similarity` no es una probabilidad.

## Conclusión

**Resumen:** R1 (StandardScaler + coseno, brute) reproduce el baseline
especificado y muestra las similitudes más altas en las inspecciones manuales.
R2–R4 quedan documentados como alternativas. La configuración inicial de
producción se fija en **R1** a la espera de la aprobación del propietario
(AGENTS.md §7 y §30).

Limitaciones: las semillas son inspección manual, no ground truth; la cobertura
completa y la latencia p50/p95 se miden en la evaluación offline.